# Phase 1: Exploratory Data Analysis (EDA)

Goal: Load, understand, and clean the crude oil dataset.

In [ ]:
# Install dependencies
!pip install -q pandas numpy matplotlib seaborn statsmodels ta

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')

## 1. Load Data

In [ ]:
df = pd.read_csv('Crude_Oil.csv', parse_dates=['Date'])
df.set_index('Date', inplace=True)
df.sort_index(inplace=True)

print(f'Shape: {df.shape}')
print(f'Date range: {df.index.min().date()} → {df.index.max().date()}')
df.head()

## 2. Basic Info

In [ ]:
df.info()

## 3. Missing Values Check

In [ ]:
print('Missing values:')
print(df.isnull().sum())
print(f'\nZero-volume days: {(df["Volume"] == 0).sum()}')

## 4. Statistical Summary

In [ ]:
df.describe()

## 5. Outlier Detection (Z-Score)

In [ ]:
from scipy import stats

outlier_df = df.copy()
for col in ['Daily_Return_%', 'Volume']:
    z = np.abs(stats.zscore(df[col]))
    outlier_df[f'{col}_outlier'] = z > 5
    outliers = df[z > 5]
    print(f'{col}: {len(outliers)} outliers (Z > 5)')
    if len(outliers) > 0:
        print(outliers[[col]].head())
print(f'\nTotal outliers flagged: {outlier_df[["Daily_Return_%_outlier", "Volume_outlier"]].any(axis=1).sum()}')

## 6. Price Chart

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(df.index, df['Close'], linewidth=0.8)
plt.title('Crude Oil Close Price (2000–Present)')
plt.xlabel('Date')
plt.ylabel('Price ($)')
plt.show()

## 7. Volume Chart

In [ ]:
plt.figure(figsize=(14, 3))
plt.bar(df.index, df['Volume'], width=1, alpha=0.7)
plt.title('Trading Volume')
plt.ylabel('Volume')
plt.show()

## 8. Stationarity Test (ADF)

In [ ]:
def adf_test(series, name):
    result = adfuller(series.dropna(), autolag='AIC')
    print(f'{name}:')
    print(f'  ADF Stat = {result[0]:.4f}, p-value = {result[1]:.4f}')
    print(f'  Stationary: {"Yes" if result[1] < 0.05 else "No"}')

adf_test(df['Close'], 'Close')
adf_test(df['Daily_Return_%'], 'Daily_Return_%')

## 9. Time Series Decomposition (Annual)

In [ ]:
decomp = seasonal_decompose(df['Close'], model='additive', period=252)
fig, axes = plt.subplots(4, 1, figsize=(14, 8))
decomp.observed.plot(ax=axes[0], title='Observed')
decomp.trend.plot(ax=axes[1], title='Trend')
decomp.seasonal.plot(ax=axes[2], title='Seasonality')
decomp.resid.plot(ax=axes[3], title='Residual')
plt.tight_layout()
plt.show()

## 10. Correlation Matrix

In [ ]:
cols = ['Open', 'High', 'Low', 'Close', 'Volume', 'Intraday_Volatility']
plt.figure(figsize=(8, 6))
sns.heatmap(df[cols].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()

## 11. Key Findings Summary

- Dataset spans from 2000 to present (~6,400 rows)
- No missing values in core price columns
- Black swan outliers present (2008 crisis, 2020 negative prices) — flagged, NOT removed
- Close price is non-stationary (ADF p > 0.05) — expected for price series
- Daily_Return_% is stationary (ADF p < 0.05)
- High OHL correlation expected for OHLC data

### Next: Proceed to Phase 2 → Feature Engineering